# 01 · HMDA 数据理解与准备

这是项目的**数据入口**。从头运行后，在 `artifacts/prepared/` 生成统一数据、字段说明、固定五折和校验清单，再运行 `02_hmda_models.ipynb`。

本 notebook 合并原 01/02 的数据理解与类型规范，并收录 10–14 的有效处理结论。它只做确定性的格式整理：不删申请、不猜测异常值、不使用目标构造特征，也不在全数据上拟合填补器、编码器或缩放器。原始 CSV 保持原样。

**任务口径：**预测 `accepted=1` 的概率；`id` 仅用于对齐。train/test 是项目提供的两份文件，交叉验证的训练折/验证折在 train 内划分。下列字段含义沿用项目内已有审计资料，未额外核验外部数据字典。


In [1]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis')
print("项目文件：", [p.name for p in project.iterdir()])
assert (project / 'train.csv').is_file(), "项目文件夹里没有 train.csv"
os.chdir(project)
print("当前目录：", Path.cwd())

Mounted at /content/drive
项目文件： ['sample_submission.csv', 'test.csv', 'train.csv', 'requirements.txt', 'README.md', 'artifacts', 'history', 'archive', '01_hmda_data_preparation.ipynb', '03_hmda_colab_gpu.ipynb', '02_hmda_models.ipynb']
当前目录： /content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

# 在项目根目录启动 Jupyter；在其他位置运行时显式设置这两个环境变量。
DATA_DIR = Path(os.environ.get("HMDA_DATA_DIR", str(Path.cwd()))).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("HMDA_OUTPUT_DIR", str(Path.cwd()))).expanduser().resolve()
PREPARED_DIR = OUTPUT_ROOT / "artifacts" / "prepared"
RANDOM_STATE = 42
N_SPLITS = 5
SCHEMA_VERSION = "1.0"
MISSING_CATEGORY = "MISSING"

pd.set_option("display.max_colwidth", 75)
print(f"原始数据：{DATA_DIR}\n输出目录：{PREPARED_DIR}")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | sklearn {sklearn.__version__}")


原始数据：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis
输出目录：/content/drive/MyDrive/Colab Notebooks/demandes-de-pret-immobilier-aux-etats-unis/artifacts/prepared
Python 3.13.15 | pandas 2.2.3 | sklearn 1.6.1


## 1. 读取原始文本并验证身份与标签

全部先读成文本，避免 `00123` 丢失前导零，也避免 pandas 把合法类别当成缺失。必须通过字段顺序、ID、标签和提交模板检查。标签在转整数**之前**检查为精确的 `0`/`1`，防止 `0.9` 被截断成 `0`。


In [3]:
SOURCE_FILES = ["train.csv", "test.csv", "sample_submission.csv"]
for name in SOURCE_FILES:
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"缺少 {DATA_DIR / name}；请修改 DATA_DIR 或 HMDA_DATA_DIR。")

def read_text_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)

train_raw = read_text_csv(DATA_DIR / "train.csv")
test_raw = read_text_csv(DATA_DIR / "test.csv")
sample_raw = read_text_csv(DATA_DIR / "sample_submission.csv")
assert {"id", "accepted"}.issubset(train_raw.columns)
assert "id" in test_raw and "accepted" not in test_raw
assert sample_raw.columns.tolist() == ["id", "accepted"], "提交模板字段变化，先核查接口"
feature_columns = train_raw.columns.drop(["id", "accepted"]).tolist()
assert feature_columns == test_raw.columns.drop("id").tolist(), "训练/测试特征或顺序不一致"
assert len(train_raw) > 0 and len(test_raw) > 0
for name, frame in [("train", train_raw), ("test", test_raw), ("sample", sample_raw)]:
    assert frame.columns.is_unique, f"{name}: 重复列名"
    assert frame["id"].str.strip().ne("").all() and frame["id"].is_unique, f"{name}: ID 无效"
    assert frame["id"].eq(frame["id"].str.strip()).all(), f"{name}: ID 有首尾空格"
assert set(train_raw["id"]).isdisjoint(test_raw["id"]), "train/test ID 重叠"
assert set(sample_raw["id"]) == set(test_raw["id"]), "提交模板与测试 ID 不匹配"
assert train_raw["accepted"].isin(["0", "1"]).all(), "accepted 必须为 0 或 1"
y = train_raw["accepted"].astype("int8")
assert set(y.unique()) == {0, 1}, "二分类训练数据必须包含两个类别"
train_labels = pd.DataFrame({"id": train_raw["id"], "accepted": y})
test_ids = test_raw[["id"]].copy()
display(pd.DataFrame({
    "数据": ["train", "test"], "行数": [len(train_raw), len(test_raw)],
    "特征列": [len(feature_columns)] * 2,
}))
display(y.value_counts().sort_index().rename("数量").to_frame().assign(比例=lambda t: t["数量"] / len(y)))


,数据,行数,特征列
0,train,105000,56
1,test,45000,56


,数量,比例
accepted,,
0,15756,0.150057
1,89244,0.849943


## 2. 明确字段类型与处理边界

13 个数值字段保持原单位，其他字段按类别处理，包括数字类别代码、DTI 区间、年龄区间和地理代码。保留完整 56 列基础特征；选择 N2 地理简化、交互特征和 LTV 标记属于模型 notebook。

| 输入情况 | 统一处理 |
|---|---|
| 数值空白（含首尾空格） | `NaN`，数值文本转换失败时直接报错 |
| 类别空白 | 字符串 `MISSING` |
| 整数类别代码 `1.0` / `1.00` | 规范为 `1`，不修改区间标签 |
| `county_code` / `derived_msa-md` | 五位字符串；保留 `99999` 特殊类别 |
| 不适用、无共同申请人、豁免等代码 | 保留各自原代码，与空白分开 |
| 负收入、零值、大额、LTV 极端值、房龄 -1 | 保留并审计，不自动删除或替换 |


In [4]:
numeric_columns = [
    "loan_amount", "loan_to_value_ratio", "property_value", "income", "loan_term",
    "total_units", "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage",
    "tract_owner_occupied_units", "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units",
]
categorical_columns = [c for c in feature_columns if c not in numeric_columns]
assert len(feature_columns) == 56, "输入 schema 变化；先审核新增/删除字段"
assert len(numeric_columns) == 13 and len(categorical_columns) == 43
assert set(numeric_columns).issubset(feature_columns)
assert set(numeric_columns).isdisjoint(categorical_columns)

schema = {
    "schema_version": SCHEMA_VERSION,
    "target_column": "accepted", "id_column": "id",
    "feature_columns": feature_columns,
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "numeric_dtype": "float64", "categorical_dtype": "object",
    "missing_category": MISSING_CATEGORY,
    "csv_read_options": {"dtype": "str", "keep_default_na": False},
    "geographic_columns": ["county_code", "derived_msa-md"],
    "row_alignment": "CSV row order; train_labels.id and test_ids.id are the authoritative IDs",
    "folds": {"n_splits": N_SPLITS, "shuffle": True, "random_state": RANDOM_STATE, "fold_values": [1, 2, 3, 4, 5]},
}


## 3. 使用同一个确定性函数处理 train/test

函数不接收标签、不计算中位数或分位数。全部数值仍可缺失，类别均为非空字符串，因此原生类别模型和 sklearn Pipeline 都能在此基础上各自处理。


In [5]:
def prepare_features(raw_features):
    assert raw_features.columns.tolist() == feature_columns, "特征或列顺序变化"
    prepared = raw_features.copy()
    for column in numeric_columns:
        values = prepared[column].str.strip().replace("", np.nan)
        prepared[column] = pd.to_numeric(values, errors="raise").astype("float64")
        assert not np.isinf(prepared[column].to_numpy()).any(), f"{column}: 存在无穷值"
        assert prepared[column].isna().equals(raw_features[column].str.strip().eq("")), f"{column}: 意外缺失"
    for column in categorical_columns:
        values = prepared[column].str.strip()
        assert not values.eq(MISSING_CATEGORY).any(), f"{column}: 原始类别与缺失标记冲突"
        values = values.str.replace(r"^(\d+)\.0+$", r"\1", regex=True)
        if column in schema["geographic_columns"]:
            nonblank = values.ne("")
            assert values[nonblank].str.fullmatch(r"\d{1,5}").all(), f"{column}: 地理代码格式异常"
            values = values.where(~nonblank, values.str.zfill(5))
        prepared[column] = values.replace("", MISSING_CATEGORY).astype(object)
    return prepared

X_train = prepare_features(train_raw[feature_columns])
X_test = prepare_features(test_raw[feature_columns])
print("格式整理完成；原始数值仍保持原单位，未进行填补、缩放或编码。")


格式整理完成；原始数值仍保持原单位，未进行填补、缩放或编码。


## 4. 验证信息和行顺序没有丢失

检查训练标签、测试 ID 与特征逐行对应，数值保真，类别缺失标记只替代空白。未见类别保留原值；后续 One-Hot 使用 `handle_unknown='ignore'`，原生类别模型按其接口处理。


In [6]:
for name, raw, features in [("train", train_raw, X_train), ("test", test_raw, X_test)]:
    assert features.shape == (len(raw), len(feature_columns))
    assert features.columns.tolist() == feature_columns and features.index.equals(raw.index)
    assert "id" not in features and "accepted" not in features
    assert features[numeric_columns].dtypes.eq("float64").all()
    for column in numeric_columns:
        expected = pd.to_numeric(raw[column].str.strip().replace("", np.nan), errors="raise").astype("float64")
        pd.testing.assert_series_equal(features[column], expected)
    for column in categorical_columns:
        assert features[column].map(lambda v: isinstance(v, str) and bool(v)).all()
        assert features[column].eq(MISSING_CATEGORY).equals(raw[column].str.strip().eq(""))
assert X_train.index.equals(y.index)
pd.testing.assert_series_equal(train_labels["id"], train_raw["id"])
pd.testing.assert_series_equal(test_ids["id"], test_raw["id"])

unseen_rows = []
for column in categorical_columns:
    unseen_values = sorted(set(X_test[column]) - set(X_train[column]))
    if unseen_values:
        unseen_rows.append({"字段": column, "测试新类别数": len(unseen_values),
                            "涉及测试行数": int(X_test[column].isin(unseen_values).sum()),
                            "示例": ", ".join(unseen_values[:5])})
unseen_categories = pd.DataFrame(unseen_rows, columns=["字段", "测试新类别数", "涉及测试行数", "示例"])
display(unseen_categories)
print("类型、缺失、数值保真与行对齐检查通过。")


,字段,测试新类别数,涉及测试行数,示例
0,county_code,84,115,"01023, 02050, 02063, 02220, 05013"
1,derived_loan_product_type,1,1,VA:Subordinate Lien


类型、缺失、数值保真与行对齐检查通过。


## 5. 精简数据概览

空白与特殊代码应分开理解。例如 `co-applicant_sex=5` 表示无共同申请人，`1111` 是部分字段的豁免代码；这些状态不统一转成缺失。`income` 单位是千美元/年，贷款额和房价单位是美元，LTV 是百分数。DTI 保留原始类别，暂不强行转成区间中点。

下列分位数只用于描述，**不作为模型阈值**。对测试集只检查输入分布与接口，不用它选择模型或处理规则。


In [7]:
missing_table = pd.DataFrame({
    "字段": feature_columns,
    "训练空白数": [int(train_raw[c].str.strip().eq("").sum()) for c in feature_columns],
    "测试空白数": [int(test_raw[c].str.strip().eq("").sum()) for c in feature_columns],
})
missing_table["训练空白比例"] = missing_table["训练空白数"] / len(X_train)
missing_table["测试空白比例"] = missing_table["测试空白数"] / len(X_test)
display(missing_table.query("训练空白数 > 0 or 测试空白数 > 0").sort_values("训练空白比例", ascending=False))
display(X_train[numeric_columns].describe(percentiles=[0.5, 0.99, 0.999]).T.round(3))
category_profile = pd.DataFrame({
    "字段": categorical_columns,
    "训练类别数": [X_train[c].nunique() for c in categorical_columns],
    "最大类别比例": [X_train[c].value_counts(normalize=True).iloc[0] for c in categorical_columns],
}).sort_values("训练类别数", ascending=False)
display(category_profile.head(10))


,字段,训练空白数,测试空白数,训练空白比例,测试空白比例
45,co-applicant_age,58693,25130,0.558981,0.558444
1,loan_to_value_ratio,11874,5079,0.113086,0.112867
28,debt_to_income_ratio,7679,3316,0.073133,0.073689
2,property_value,4756,1988,0.045295,0.044178
3,income,4578,2022,0.043600,0.044933
4,loan_term,3307,1397,0.031495,0.031044
44,applicant_age,3088,1354,0.029410,0.030089
8,county_code,1339,593,0.012752,0.013178
6,state_code,824,367,0.007848,0.008156
5,total_units,437,170,0.004162,0.003778


,count,mean,std,min,50%,99%,99.9%,max
loan_amount,105000.0,379120.476,1878546.887,5000.0,285000.00,1615000.000,5505080.000,3.750050e+08
loan_to_value_ratio,93126.0,6528.789,1966145.822,0.5,90.00,104.719,163.219,6.000000e+08
property_value,100244.0,515903.695,5675570.637,5000.0,355000.00,2340700.000,8461260.000,1.581075e+09
income,100422.0,154.182,352.528,-657.0,107.00,827.790,2831.948,7.100000e+04
loan_term,101693.0,337.749,66.408,1.0,360.00,364.000,480.000,4.800000e+02
total_units,104563.0,1.039,0.257,1.0,1.00,2.000,4.000,4.000000e+00
tract_population,105000.0,4759.982,2310.308,0.0,4490.00,12130.000,24651.000,3.019900e+04
tract_minority_population_percent,105000.0,36.636,25.702,0.0,29.91,98.170,99.640,1.000000e+02
ffiec_msa_md_median_family_income,105000.0,97960.353,23909.726,0.0,97500.00,164200.000,184200.000,2.013000e+05
tract_to_msa_income_percentage,105000.0,106.608,42.870,0.0,103.00,240.000,308.000,5.160000e+02


,字段,训练类别数,最大类别比例
2,county_code,2899,0.014638
1,derived_msa-md,418,0.144762
0,state_code,55,0.107752
31,co-applicant_race-1,20,0.547810
22,debt_to_income_ratio,20,0.129752
30,applicant_race-1,19,0.642924
21,co-applicant_credit_score_type,16,0.532429
20,applicant_credit_score_type,15,0.306733
27,co-applicant_ethnicity-1,10,0.548105
24,derived_race,9,0.629590


## 6. 保留风险记录，给出审计计数

原 13 的审计支持保留负收入、零收入和大额申请；LTV 高于 100% 不自动视为错误。极端 LTV 和负房龄仍是待核实事项。重复特征也可能对应不同真实申请，先报告，不能直接删除。

相同特征跨折可能使随机 CV 偏乐观；未来若得到机构、时间或客户标识，应补充相应分组或时间验证。当前固定随机五折只延续已有开发比较口径。


In [8]:
audit_rows = []
def audit(dataset, check, count, action="保留并记录"):
    audit_rows.append({"dataset": dataset, "check": check, "count": int(count), "action": action})

for name, raw, features in [("train", train_raw, X_train), ("test", test_raw, X_test)]:
    audit(name, "duplicate_ids", raw["id"].duplicated().sum(), "必须为0；已校验")
    audit(name, "duplicate_feature_rows_after_first", features.duplicated().sum())
    audit(name, "income_negative", features["income"].lt(0).sum())
    audit(name, "income_zero", features["income"].eq(0).sum())
    audit(name, "ltv_above_200_percent", features["loan_to_value_ratio"].gt(200).sum())
    audit(name, "ltv_above_1000_percent", features["loan_to_value_ratio"].gt(1000).sum(), "保留；原始来源待核实")
    audit(name, "negative_housing_age", features["tract_median_age_of_housing_units"].lt(0).sum(), "保留；特殊编码待核实")
    for column in ["loan_amount", "property_value", "loan_term"]:
        audit(name, f"{column}_nonpositive", features[column].le(0).sum(), "若非0，人工核查")
    audit(name, "minority_percent_outside_0_100", (features["tract_minority_population_percent"].lt(0) | features["tract_minority_population_percent"].gt(100)).sum(), "若非0，人工核查")
    for column in feature_columns:
        audit(name, f"blank::{column}", raw[column].str.strip().eq("").sum(), "按字段类型规范缺失")

# 哈希用于紧凑的重复组合审计，不用于删除或匹配标签。
train_feature_hash = pd.util.hash_pandas_object(X_train, index=False)
test_feature_hash = pd.util.hash_pandas_object(X_test, index=False)
audit("train_test", "shared_feature_hash_values", len(set(train_feature_hash) & set(test_feature_hash)), "保留；仅诊断，存在理论哈希碰撞可能")
for row in unseen_rows:
    audit("test", f"unseen_category_rows::{row['字段']}", row["涉及测试行数"], "保留原类别，模型处理未见类别")
audit_table = pd.DataFrame(audit_rows)
display(audit_table[~audit_table["check"].str.startswith("blank::")])


,dataset,check,count,action
0,train,duplicate_ids,0,必须为0；已校验
1,train,duplicate_feature_rows_after_first,7,保留并记录
2,train,income_negative,28,保留并记录
3,train,income_zero,926,保留并记录
4,train,ltv_above_200_percent,34,保留并记录
5,train,ltv_above_1000_percent,3,保留；原始来源待核实
6,train,negative_housing_age,5,保留；特殊编码待核实
7,train,loan_amount_nonpositive,0,若非0，人工核查
8,train,property_value_nonpositive,0,若非0，人工核查
9,train,loan_term_nonpositive,0,若非0，人工核查


## 7. 旧实验中保留什么

| 原实验 | 整理后的决定 | 放在哪里 |
|---|---|---|
| 01/02 数据理解与类型规范 | 保留字段规则、缺失概览、完整数值与类别；精简重复展示 | 当前 notebook |
| 04/05 缺失标记与金额/收入对数 | 保留 sklearn 模型的有效基础流程 | 模型 Pipeline，逐折拟合填补器 |
| 10 通用处理实验 | 不默认加财务比率、不统一裁尾，不把社区全零自动设为缺失 | 历史结论保留，默认运行不重做 |
| 11/12 N2 与 DTI×loan_type | 保留 N2 作为传统模型简化表示；交互只用于得到支持的 LR | 模型 notebook 的配置/特征构造 |
| 13 极端值审计 | 保留风险记录；不套用全局 IQR 删行 | 当前 notebook |
| 14 LTV 尾部实验 | LR 保留 P99/P99.9 标记，RF 保留原值；不默认截断 | 阈值仅从当前训练折计算 |

这些决定来自既有开发实验，不能理解为对所有未来模型都成立。原生类别模型可自行选择所需表示；当前基础数据保留全部信息。

## 8. 固定交叉验证折

使用与历史主实验一致的 `StratifiedKFold(5, shuffle=True, random_state=42)`，折号为 1–5。保存 `id,fold`，所有模型按 ID 和行顺序核对后复用。多轮选择已经使用这些折，AUC 是开发期估计；更改种子也不会自动得到未见过的独立测试集。


In [9]:
assert y.value_counts().min() >= N_SPLITS
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_assignment = np.zeros(len(y), dtype="int8")
validation_seen = np.zeros(len(y), dtype="int8")
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_train, y), start=1):
    assert len(np.intersect1d(fit_idx, valid_idx)) == 0
    assert len(fit_idx) + len(valid_idx) == len(y)
    fold_assignment[valid_idx] = fold
    validation_seen[valid_idx] += 1
assert np.all(validation_seen == 1)
assert set(fold_assignment.tolist()) == set(range(1, N_SPLITS + 1))
folds = pd.DataFrame({"id": train_labels["id"], "fold": fold_assignment})
fold_profile = train_labels.assign(fold=fold_assignment).groupby("fold")["accepted"].agg(["size", "mean"])
fold_profile.columns = ["验证行数", "验证获批率"]
display(fold_profile)


,验证行数,验证获批率
fold,,
1,21000,0.849952
2,21000,0.849952
3,21000,0.849952
4,21000,0.849952
5,21000,0.849905


## 9. 输出稳定接口与数据指纹

`train_features.csv` / `test_features.csv` 只含 56 个特征，数值缺失写为空白。`train_labels.csv` 与 `test_ids.csv` 记录对应行的 ID；`folds.csv` 记录训练 ID 的固定折号。`schema.json` 描述如何恢复类型，`manifest.json` 记录输入/输出 SHA-256 和环境信息。

模型 notebook 必须先检查这些文件的哈希、字段和逐行 ID，再开始训练。重新运行此 notebook 会刷新本目录的准备结果；原始 CSV 不会改动。


In [10]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
output_frames = {
    "train_features.csv": X_train, "test_features.csv": X_test,
    "train_labels.csv": train_labels, "test_ids.csv": test_ids,
    "folds.csv": folds, "audit.csv": audit_table,
}
for name, frame in output_frames.items():
    destination = PREPARED_DIR / name
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    frame.to_csv(temporary, index=False, na_rep="")
    temporary.replace(destination)
(PREPARED_DIR / "schema.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

manifest = {
    "schema_version": SCHEMA_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dir": str(DATA_DIR),
    "source_files": {name: {"sha256": sha256_file(DATA_DIR / name), "bytes": (DATA_DIR / name).stat().st_size} for name in SOURCE_FILES},
    "output_files": {name: {"sha256": sha256_file(PREPARED_DIR / name), "bytes": (PREPARED_DIR / name).stat().st_size} for name in [*output_frames, "schema.json"]},
    "n_rows": {"train": len(X_train), "test": len(X_test)},
    "n_features": len(feature_columns),
    "seed": RANDOM_STATE, "n_splits": N_SPLITS,
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "preprocessing": "deterministic text/type normalization only; no fitted estimator or population-derived threshold",
}
# manifest 最后写入；若中途失败，旧 manifest 与文件哈希不匹配，模型入口会停止。
manifest_temp = PREPARED_DIR / "manifest.json.tmp"
manifest_temp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
manifest_temp.replace(PREPARED_DIR / "manifest.json")
print(f"已保存 {len(output_frames) + 2} 个准备文件。")


已保存 8 个准备文件。


## 10. 回读验证

CSV 不保存 dtype，所以读取时必须先保留字符串，再按 schema 恢复数值。此处验证回读后数值、类别、列顺序和 ID 一致；不能直接对已准备数据再次调用原始输入清洗函数（其中 `MISSING` 已是合法标记）。


In [11]:
def read_prepared_features(path, saved_schema):
    frame = read_text_csv(path)
    assert frame.columns.tolist() == saved_schema["feature_columns"]
    for column in saved_schema["numeric_columns"]:
        frame[column] = pd.to_numeric(frame[column].replace("", np.nan), errors="raise").astype("float64")
    for column in saved_schema["categorical_columns"]:
        assert frame[column].ne("").all(), f"{column}: 准备文件出现空白类别"
        frame[column] = frame[column].astype(object)
    return frame

saved_schema = json.loads((PREPARED_DIR / "schema.json").read_text(encoding="utf-8"))
for name, expected in [("train_features.csv", X_train), ("test_features.csv", X_test)]:
    restored = read_prepared_features(PREPARED_DIR / name, saved_schema)
    pd.testing.assert_frame_equal(restored, expected, check_exact=False, rtol=1e-14, atol=0)
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "train_labels.csv"), train_labels.astype(str))
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "test_ids.csv"), test_ids)
pd.testing.assert_frame_equal(read_text_csv(PREPARED_DIR / "folds.csv"), folds.astype(str))
for name, entry in manifest["output_files"].items():
    assert sha256_file(PREPARED_DIR / name) == entry["sha256"], f"输出哈希变化：{name}"
print("准备文件回读、类型恢复、行对齐与 SHA-256 检查全部通过。")


准备文件回读、类型恢复、行对齐与 SHA-256 检查全部通过。
